In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm          # normal distribution: norm.pdf(x) is the N(0,1) density
from pathlib import Path

FIG_DIR = Path("report/figures")      # relative to this notebook's folder (project1/)
FIG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(2503)

In [3]:
import yfinance as yf
import pandas as pd

DATA_FILE = Path("data/nvda_close.csv")   # relative to project1/, like FIG_DIR
REFRESH = False                           # set True ONLY when you deliberately want a new price

def fetch_last_close(ticker="NVDA"):
    hist = yf.Ticker(ticker).history(period="5d", auto_adjust=False)
    return pd.DataFrame({
        "date":   [hist.index[-1].date()],             # date of the most recent row
        "close":  [round(float(hist["Close"].iloc[-1]), 2)],
        "source": ["Yahoo Finance (yfinance)"],
    })

if REFRESH or not DATA_FILE.exists():     
    fetch_last_close().to_csv(DATA_FILE, index=False)

price = pd.read_csv(DATA_FILE)
S0 = float(price["close"].iloc[0])
K  = S0
print(price)

         date   close                    source
0  2026-09-30  228.38  Yahoo Finance (yfinance)


In [8]:
T     = 1.0     # years
r     = 0.04    # risk-free rate (continuous compounding)
sigma = 0.30    # annualised volatility

def crr_parameters(sigma, r, T, N):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    growth = np.exp(r * dt)
    if not (d < growth < u):
        raise ValueError("No-arbitrage condition d < e^{r dt} < u fails")
    p = (growth - d) / (u - d)

    return dt, u, d, p

In [13]:

def prices_at_step(S0, u, d, n): 
    j = np.arange(n+1)
    S_n = S0 * u**j * d**(n-j)
    return S_n

dt, u, d, p = crr_parameters(sigma, r, T, 3)
for n in range(4):
    print(n, np.round(prices_at_step(S0, u, d, n), 2))

0 [228.38]
1 [192.06 271.57]
2 [161.52 228.38 322.93]
3 [135.83 192.06 271.57 383.99]


In [16]:
def call_payoff(S, K):
    payoff = np.maximum(S-K, 0)
    return payoff

In [18]:
def backward_induction(V_T, p, disc_fctr): 
    V = V_T.copy()
    q = 1 - p
    for i in range(len(V_T) - 1):
        V = disc_fctr * ( p * V[1:] + q * V[:-1])
    return V[0]

In [19]:
V_T = call_payoff(prices_at_step(4, 2, 0.5, 3), 6)
print(V_T)                                               # expect [ 0.  0.  2. 26.]
print(backward_induction(V_T, p=0.5, disc_fctr=1/1.25))  # expect 2.048

[ 0.  0.  2. 26.]
2.0480000000000005
